In [2]:
print("hello world")

hello world


In [ ]:

import serial
import time
import csv
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

# Change this to your Arduino's serial port
PORT = "/dev/cu.usbmodem1101"  # Example for Mac
BAUD_RATE = 115200
SAMPLE_RATE = 100  # Assumed 100 samples per second
WINDOW_SIZE = 500  # Analyze the most recent 5 seconds

# Connect to Arduino
arduino = serial.Serial(PORT, BAUD_RATE, timeout=1)
time.sleep(2)  # Allow Arduino to reset

timestamps = deque(maxlen=WINDOW_SIZE)
accel_x = deque(maxlen=WINDOW_SIZE)
accel_y = deque(maxlen=WINDOW_SIZE)
accel_z = deque(maxlen=WINDOW_SIZE)

start_time = time.perf_counter()

plt.ion()
fig, ax = plt.subplots()
line_x, = ax.plot([], [], label="X acceleration")
line_y, = ax.plot([], [], label="Y acceleration")
line_z, = ax.plot([], [], label="Z acceleration")

ax.set_xlabel("Time (s)")
ax.set_ylabel("Acceleration (sensor units)")
ax.set_title("Live IMU Vibration")
ax.legend()

try:
    with open("imu_data.csv", "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["time_s", "ax", "ay", "az"])

        while True:
            raw = arduino.readline().decode(
                "utf-8", errors="ignore"
            ).strip()

            try:
                values = [float(v) for v in raw.split(",")]
                if len(values) != 3:
                    continue

                ax_val, ay_val, az_val = values
                t = time.perf_counter() - start_time

                timestamps.append(t)
                accel_x.append(ax_val)
                accel_y.append(ay_val)
                accel_z.append(az_val)

                writer.writerow([t, ax_val, ay_val, az_val])

                if len(timestamps) == WINDOW_SIZE:
                    # Remove the mean to focus on changing motion
                    signal = np.array(accel_x)
                    signal = signal - np.mean(signal)

                    frequencies = np.fft.rfftfreq(
                        len(signal), d=1 / SAMPLE_RATE
                    )
                    spectrum = np.abs(np.fft.rfft(signal))

                    # Ignore zero frequency
                    spectrum[0] = 0
                    dominant_frequency = frequencies[
                        np.argmax(spectrum)
                    ]

                    print(
                        f"Dominant X-axis frequency: "
                        f"{dominant_frequency:.2f} Hz"
                    )

                # Update live plot
                line_x.set_data(timestamps, accel_x)
                line_y.set_data(timestamps, accel_y)
                line_z.set_data(timestamps, accel_z)

                ax.relim()
                ax.autoscale_view()
                plt.pause(0.001)

            except ValueError:
                # Ignore incomplete or non-numeric lines
                continue

except KeyboardInterrupt:
    print("Stopping data collection...")

finally:
    arduino.close()
    plt.ioff()
    plt.show()
